# Aula 14 — Gradient checking

Este laboratório transforma a verificação pontual da aula anterior em um protocolo auditável. Implementaremos diferenças para frente e centrais, erro relativo, varredura do passo, checks coordenados e direcionais, além de contraprovas com bug, ReLU e aleatoriedade.

**Dependências:** Python ≥ 3.11, NumPy ≥ 1.26, Matplotlib ≥ 3.8 e nbformat ≥ 5.9 para validação.  
**Seed:** `20260914`.  
**Escopo:** NumPy puro, `float64` por padrão, dados sintéticos e nenhuma credencial ou download.

In [ ]:
import platform
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

SEED = 20260914
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=9, suppress=True)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seed:", SEED)

## 1. MLP de referência

Usaremos exemplos nas linhas, uma camada oculta `tanh` e softmax com cross-entropy média. O forward estável e o backward são independentes do verificador numérico.

In [ ]:
def validate(X, Y, params):
    X, Y = np.asarray(X), np.asarray(Y)
    W1, b1, W2, b2 = (np.asarray(params[k]) for k in ("W1", "b1", "W2", "b2"))
    assert X.ndim == Y.ndim == W1.ndim == W2.ndim == 2
    assert b1.ndim == b2.ndim == 1
    assert X.shape[0] == Y.shape[0] > 0
    assert X.shape[1] == W1.shape[0]
    assert W1.shape[1] == b1.shape[0] == W2.shape[0]
    assert W2.shape[1] == b2.shape[0] == Y.shape[1]
    assert np.allclose(Y.sum(axis=1), 1.0)
    assert all(np.all(np.isfinite(a)) for a in (X, Y, W1, b1, W2, b2))


def forward(X, Y, params):
    validate(X, Y, params)
    X, Y = np.asarray(X), np.asarray(Y)
    W1, b1, W2, b2 = (params[k] for k in ("W1", "b1", "W2", "b2"))
    Z1 = X @ W1 + b1
    A1 = np.tanh(Z1)
    Z2 = A1 @ W2 + b2
    maximum = Z2.max(axis=1, keepdims=True)
    logsumexp = maximum + np.log(np.exp(Z2 - maximum).sum(axis=1, keepdims=True))
    log_probs = Z2 - logsumexp
    P = np.exp(log_probs)
    loss = -np.sum(Y * log_probs) / X.shape[0]
    cache = {k: np.asarray(v).copy() for k, v in {
        "X": X, "Y": Y, "W1": W1, "b1": b1, "A1": A1,
        "W2": W2, "b2": b2, "P": P,
    }.items()}
    assert np.isfinite(loss) and np.allclose(P.sum(axis=1), 1.0)
    return float(loss), cache


def backward(cache, omit_tanh=False):
    X, Y, W1, A1, W2, P = (cache[k] for k in ("X", "Y", "W1", "A1", "W2", "P"))
    G2 = (P - Y) / X.shape[0]
    dW2 = A1.T @ G2
    db2 = G2.sum(axis=0)
    dA1 = G2 @ W2.T
    G1 = dA1 if omit_tanh else dA1 * (1.0 - A1**2)
    grads = {
        "W1": X.T @ G1,
        "b1": G1.sum(axis=0),
        "W2": dW2,
        "b2": db2,
    }
    assert all(grads[k].shape == cache[k].shape for k in grads)
    return grads


X = rng.normal(size=(5, 3)).astype(np.float64)
labels = np.array([0, 2, 1, 2, 0])
Y = np.eye(3, dtype=np.float64)[labels]
params = {
    "W1": rng.normal(0, 0.3, size=(3, 4)),
    "b1": rng.normal(0, 0.1, size=4),
    "W2": rng.normal(0, 0.3, size=(4, 3)),
    "b2": rng.normal(0, 0.1, size=3),
}
loss, cache = forward(X, Y, params)
grads = backward(cache)
print(f"Loss de referência: {loss:.9f}")
print("Parâmetros:", sum(v.size for v in params.values()))

## 2. Empacotamento reversível

O checker percorre um vetor, mas preserva nome, shape, dtype e intervalo de cada tensor. `unpack(pack(params))` deve reconstruir todos os valores sem compartilhar memória com o original.

In [ ]:
PARAMETER_ORDER = ("W1", "b1", "W2", "b2")


def make_spec(mapping):
    spec, start = [], 0
    for name in PARAMETER_ORDER:
        value = np.asarray(mapping[name])
        stop = start + value.size
        spec.append({"name": name, "shape": value.shape, "start": start, "stop": stop, "dtype": value.dtype.str})
        start = stop
    return spec


def pack(mapping, spec):
    return np.concatenate([np.asarray(mapping[item["name"]]).reshape(-1) for item in spec]).copy()


def unpack(vector, spec):
    vector = np.asarray(vector)
    assert vector.ndim == 1 and vector.size == spec[-1]["stop"]
    return {
        item["name"]: vector[item["start"]:item["stop"]].reshape(item["shape"]).copy()
        for item in spec
    }


spec = make_spec(params)
theta = pack(params, spec)
gradient = pack(grads, spec)
roundtrip = unpack(theta, spec)
assert theta.shape == gradient.shape == (31,)
assert all(np.array_equal(roundtrip[k], params[k]) for k in PARAMETER_ORDER)
assert all(not np.shares_memory(roundtrip[k], theta) for k in PARAMETER_ORDER)
print("Vetor empacotado:", theta.shape)
for item in spec:
    print(item)

## 3. Diferenças e métricas de erro

O passo relativo é (h_j=h_0\max(1,|\theta_j|)). A diferença central refaz o forward completo em duas cópias. Reportamos erro absoluto e erro relativo com piso seguro.

In [ ]:
def loss_from_vector(vector, dtype=np.float64):
    converted = np.asarray(vector, dtype=dtype)
    mapped = {k: v.astype(dtype) for k, v in unpack(converted, spec).items()}
    return forward(X.astype(dtype), Y.astype(dtype), mapped)[0]


def forward_difference(loss_fn, vector, index, base_step=1e-5):
    vector = np.asarray(vector)
    h = base_step * max(1.0, abs(float(vector[index])))
    plus = vector.copy()
    plus[index] += h
    return (loss_fn(plus) - loss_fn(vector)) / h, h


def central_difference(loss_fn, vector, index, base_step=1e-5):
    vector = np.asarray(vector)
    h = base_step * max(1.0, abs(float(vector[index])))
    plus, minus = vector.copy(), vector.copy()
    plus[index] += h
    minus[index] -= h
    return (loss_fn(plus) - loss_fn(minus)) / (2.0 * h), h


def error_metrics(analytic, numeric, floor=1e-12):
    absolute = abs(float(analytic) - float(numeric))
    relative = absolute / max(floor, abs(float(analytic)), abs(float(numeric)))
    return absolute, relative


assert loss_from_vector(theta) == loss
assert error_metrics(0.0, 0.0) == (0.0, 0.0)
print("Precisão de máquina float64:", np.finfo(np.float64).eps)
print("Precisão de máquina float32:", np.finfo(np.float32).eps)

## 4. Check exaustivo

Como há apenas 31 parâmetros, verificamos todas as coordenadas. O relatório mantém nome e índice local para localizar qualquer falha.

In [ ]:
def exhaustive_check(vector, analytic, base_step=1e-5):
    rows = []
    for item in spec:
        for flat_local in range(item["stop"] - item["start"]):
            global_index = item["start"] + flat_local
            numeric, h = central_difference(loss_from_vector, vector, global_index, base_step)
            absolute, relative = error_metrics(analytic[global_index], numeric)
            local_index = np.unravel_index(flat_local, item["shape"])
            rows.append({"tensor": item["name"], "index": local_index, "analytic": analytic[global_index],
                         "numeric": numeric, "h": h, "absolute": absolute, "relative": relative})
    return rows


rows = exhaustive_check(theta, gradient)
worst = max(rows, key=lambda row: row["relative"])
assert len(rows) == theta.size
assert worst["relative"] < 1e-7
print("Coordenadas verificadas:", len(rows))
print("Pior caso:", worst)
for name in PARAMETER_ORDER:
    group = [r for r in rows if r["tensor"] == name]
    print(f"{name}: máximo relativo={max(r['relative'] for r in group):.3e}")

## 5. Ordem de truncamento em uma função escalar

Antes do arredondamento dominar, dividir (h) por 10 reduz o erro da diferença para frente aproximadamente 10 vezes e o da central aproximadamente 100 vezes. Usamos (f(x)=\exp(x)), cuja derivada é conhecida.

In [ ]:
x0 = 0.7
truth = np.exp(x0)
orders_h = np.logspace(-1, -5, 5)
forward_errors, central_errors = [], []
for h in orders_h:
    forward_value = (np.exp(x0 + h) - np.exp(x0)) / h
    central_value = (np.exp(x0 + h) - np.exp(x0 - h)) / (2 * h)
    forward_errors.append(abs(forward_value - truth))
    central_errors.append(abs(central_value - truth))

ratio_forward = forward_errors[0] / forward_errors[1]
ratio_central = central_errors[0] / central_errors[1]
assert 9 < ratio_forward < 12
assert 90 < ratio_central < 110
print(f"Razão de melhora para frente: {ratio_forward:.3f}")
print(f"Razão de melhora central: {ratio_central:.3f}")

## 6. Varredura do passo e do dtype

O gráfico mostra o maior erro relativo da MLP para vários passos. Em `float64`, surge uma faixa estável antes do cancelamento; `float32` perde precisão mais cedo.

**Texto alternativo do gráfico:** duas curvas em escala log-log relacionam passo e maior erro relativo; a curva de float64 desce a valores muito menores, enquanto a de float32 volta a subir ou fica imprecisa para passos pequenos.

In [ ]:
steps = np.logspace(-1, -11, 11)
errors64, errors32 = [], []
for base_step in steps:
    numeric64 = np.array([central_difference(loss_from_vector, theta, i, base_step)[0] for i in range(theta.size)])
    err64 = max(error_metrics(gradient[i], numeric64[i])[1] for i in range(theta.size))
    errors64.append(err64)

    theta32 = theta.astype(np.float32)
    numeric32 = np.array([
        central_difference(lambda v: loss_from_vector(v, np.float32), theta32, i, base_step)[0]
        for i in range(theta.size)
    ])
    err32 = max(error_metrics(gradient[i], numeric32[i])[1] for i in range(theta.size))
    errors32.append(err32)

errors64, errors32 = np.array(errors64), np.array(errors32)
best64, best32 = int(np.argmin(errors64)), int(np.argmin(errors32))
assert errors64[best64] < 1e-7
assert errors32[best32] > errors64[best64] * 100

fig, ax = plt.subplots(figsize=(7.4, 4.5))
ax.loglog(steps, errors64, "o-", label="float64")
ax.loglog(steps, errors32, "s--", label="float32")
ax.invert_xaxis()
ax.set(xlabel="passo base h₀", ylabel="maior erro relativo",
       title="Truncamento e arredondamento na escolha do passo")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
print(f"Melhor float64: h={steps[best64]:.0e}, erro={errors64[best64]:.3e}")
print(f"Melhor float32: h={steps[best32]:.0e}, erro={errors32[best32]:.3e}")

## 7. Comparação para frente × central na MLP

Com o mesmo passo, a diferença central deve ser substancialmente mais precisa para coordenadas suaves.

In [ ]:
probe = int(np.argmax(np.abs(gradient)))
forward_num, _ = forward_difference(loss_from_vector, theta, probe, 1e-4)
central_num, _ = central_difference(loss_from_vector, theta, probe, 1e-4)
forward_error = error_metrics(gradient[probe], forward_num)[1]
central_error = error_metrics(gradient[probe], central_num)[1]
assert central_error < forward_error / 100
print(f"Coordenada global: {probe}")
print(f"Erro para frente: {forward_error:.3e}")
print(f"Erro central: {central_error:.3e}")
print(f"Ganho de precisão: {forward_error / central_error:.1f}×")

## 8. Amostragem estratificada

Em modelos maiores, sorteamos coordenadas dentro de cada tensor, garantindo cobertura de pesos e biases. A seed torna a amostra reprodutível.

In [ ]:
def stratified_indices(specification, per_tensor, generator):
    selected = []
    for item in specification:
        candidates = np.arange(item["start"], item["stop"])
        size = min(per_tensor, candidates.size)
        selected.extend(generator.choice(candidates, size=size, replace=False).tolist())
    return np.array(selected, dtype=int)


sample_rng = np.random.default_rng(SEED + 1)
sampled = stratified_indices(spec, per_tensor=3, generator=sample_rng)
sample_errors = []
for index in sampled:
    numeric, _ = central_difference(loss_from_vector, theta, int(index), 1e-5)
    sample_errors.append(error_metrics(gradient[index], numeric)[1])

assert sampled.size == 12
assert len(np.unique(sampled)) == sampled.size
assert max(sample_errors) < 1e-7
assert set(next(i["name"] for i in spec if i["start"] <= j < i["stop"]) for j in sampled) == set(PARAMETER_ORDER)
print("Índices globais:", sampled)
print(f"Maior erro amostrado: {max(sample_errors):.3e}")

## 9. Testes direcionais

Uma direção unitária custa apenas dois forwards. Cinco direções independentes comparam (d^\top g) com a inclinação numérica ao longo de (d).

In [ ]:
def directional_check(loss_fn, vector, analytic, direction, step=1e-5):
    direction = np.asarray(direction, dtype=np.float64)
    direction = direction / np.linalg.norm(direction)
    numeric = (loss_fn(vector + step * direction) - loss_fn(vector - step * direction)) / (2 * step)
    expected = float(analytic @ direction)
    absolute, relative = error_metrics(expected, numeric)
    return expected, numeric, absolute, relative


directional_rows = []
for _ in range(5):
    direction = rng.normal(size=theta.size)
    directional_rows.append(directional_check(loss_from_vector, theta, gradient, direction))

directional_max = max(row[3] for row in directional_rows)
assert directional_max < 1e-7
for index, row in enumerate(directional_rows, 1):
    print(f"Direção {index}: analítico={row[0]:+.9f}, numérico={row[1]:+.9f}, relativo={row[3]:.3e}")

## 10. Defeito injetado e localização

Removemos propositalmente (1-A_1^2) do backward. A camada de saída continua correta, enquanto os gradientes anteriores à `tanh` falham. Isso testa o próprio verificador.

In [ ]:
buggy_grads = backward(cache, omit_tanh=True)
buggy_gradient = pack(buggy_grads, spec)
bug_report = {}
for item in spec:
    group = [r for r in rows if r["tensor"] == item["name"]]
    rels = []
    for row in group:
        flat_local = np.ravel_multi_index(row["index"], item["shape"])
        global_index = item["start"] + flat_local
        rels.append(error_metrics(buggy_gradient[global_index], row["numeric"])[1])
    bug_report[item["name"]] = max(rels)

assert bug_report["W1"] > 0.05 and bug_report["b1"] > 0.05
assert bug_report["W2"] < 1e-7 and bug_report["b2"] < 1e-7
for name, value in bug_report.items():
    print(f"{name}: erro máximo com bug={value:.3e}")
print("Primeira fronteira divergente: backward da ativação oculta.")

## 11. Quina da ReLU

Em zero, a derivada clássica não existe. A implementação adota subgradiente 0, enquanto a secante central que atravessa a quina retorna 0,5. Fora da quina, o check volta a concordar.

In [ ]:
def relu(value):
    return np.maximum(0.0, value)


step = 1e-5
numeric_at_kink = (relu(step) - relu(-step)) / (2 * step)
analytic_convention = 0.0
x_smooth = 0.3
numeric_smooth = (relu(x_smooth + step) - relu(x_smooth - step)) / (2 * step)

assert numeric_at_kink == 0.5
assert analytic_convention == 0.0
assert np.isclose(numeric_smooth, 1.0)
print("ReLU em zero — analítico convencional:", analytic_convention)
print("ReLU em zero — central:", numeric_at_kink)
print("ReLU em 0,3 — central:", numeric_smooth)

## 12. Aleatoriedade variável versus máscara congelada

Definimos (J(w)=\frac12\|m\odot(wx)-y\|^2). Se cada avaliação sorteia uma máscara diferente, a diferença mede também o ruído. Com a mesma máscara, o gradiente numérico verifica a função correta.

In [ ]:
x_drop = np.array([0.4, -1.2, 0.7, 1.5])
y_drop = np.array([0.1, 0.3, -0.2, 0.9])
w_drop = 0.8
mask_fixed = np.array([1.0, 0.0, 1.0, 1.0])


def masked_loss(w, mask):
    residual = mask * (w * x_drop) - y_drop
    return 0.5 * float(residual @ residual)


analytic_fixed = float((mask_fixed * x_drop) @ (mask_fixed * (w_drop * x_drop) - y_drop))
numeric_fixed = (masked_loss(w_drop + 1e-5, mask_fixed) - masked_loss(w_drop - 1e-5, mask_fixed)) / 2e-5

noise_rng = np.random.default_rng(SEED + 1)
mask_plus = noise_rng.binomial(1, 0.7, size=x_drop.size)
mask_minus = noise_rng.binomial(1, 0.7, size=x_drop.size)
numeric_variable = (masked_loss(w_drop + 1e-5, mask_plus) - masked_loss(w_drop - 1e-5, mask_minus)) / 2e-5

fixed_error = error_metrics(analytic_fixed, numeric_fixed)[1]
variable_error = error_metrics(analytic_fixed, numeric_variable)[1]
assert fixed_error < 1e-10
assert variable_error > 0.1
assert not np.array_equal(mask_plus, mask_minus)
print(f"Erro com máscara congelada: {fixed_error:.3e}")
print(f"Erro com máscaras diferentes: {variable_error:.3e}")
print("Máscara +:", mask_plus, "Máscara -:", mask_minus)

## 13. Determinismo e cópias

Duas chamadas no mesmo ponto precisam retornar a mesma loss. Também verificamos que diferenças numéricas não alteraram `theta` nem os parâmetros originais.

In [ ]:
loss_repeat_1 = loss_from_vector(theta)
loss_repeat_2 = loss_from_vector(theta)
theta_after_checks = pack(params, spec)

assert loss_repeat_1 == loss_repeat_2 == loss
assert np.array_equal(theta_after_checks, theta)
assert all(np.array_equal(params[k], cache[k]) for k in PARAMETER_ORDER)
assert all(np.isfinite([row["numeric"] for row in rows]))
print("Loss repetida bit a bit:", loss_repeat_1 == loss_repeat_2)
print("Ponto-base permaneceu imutável:", np.array_equal(theta_after_checks, theta))

## 14. Auditoria final

Os grupos abaixo cobrem integridade estrutural, check completo, escala do passo, dtype, amostragem, direções, localização de bug, quina, aleatoriedade e imutabilidade.

In [ ]:
audit = {
    "loss_finita": np.isfinite(loss),
    "pack_roundtrip": all(np.array_equal(roundtrip[k], params[k]) for k in PARAMETER_ORDER),
    "31_coordenadas": len(rows) == 31,
    "check_exaustivo": worst["relative"] < 1e-7,
    "ordem_forward": 9 < ratio_forward < 12,
    "ordem_central": 90 < ratio_central < 110,
    "float64_preciso": errors64[best64] < 1e-7,
    "float32_limitado": errors32[best32] > errors64[best64] * 100,
    "central_superior": central_error < forward_error / 100,
    "amostra_estratificada": sampled.size == 12 and max(sample_errors) < 1e-7,
    "cinco_direcoes": len(directional_rows) == 5 and directional_max < 1e-7,
    "bug_detectado_W1": bug_report["W1"] > 0.05,
    "saida_preservada_no_bug": bug_report["W2"] < 1e-7,
    "quina_identificada": numeric_at_kink == 0.5,
    "mascara_congelada": fixed_error < 1e-10,
    "ruido_detectado": variable_error > 0.1,
    "determinismo": loss_repeat_1 == loss_repeat_2,
    "imutabilidade": np.array_equal(theta_after_checks, theta),
}
assert len(audit) == 18 and all(audit.values())
for name, passed in audit.items():
    print(f"[{'OK' if passed else 'FALHA'}] {name}")
print(f"\n{sum(audit.values())}/{len(audit)} grupos de auditoria aprovados.")

## Conclusões

- A diferença central verificou as 31 coordenadas da MLP.
- A varredura de (h) expôs truncamento e cancelamento.
- `float64` alcançou precisão muito superior a `float32`.
- Amostra estratificada e cinco direções reduziram custo mantendo cobertura.
- O defeito injetado falhou somente antes da `tanh`, localizando a fronteira.
- A quina da ReLU produziu divergência legítima.
- Máscaras diferentes quebraram o teste; máscara congelada restaurou a concordância.

Na Aula 15, gradientes verificados serão usados para estudar simetria e inicializações Xavier/Glorot e He/Kaiming.